# 00 Baseline Diagnosis：建立三档难度的可靠基线

正式任务为 **Numeral System（Easy）→ Gravity（Medium）→ Equation Numeric（Hard）**。
Bit Manipulation 保留在历史实验与高级扩展中。

本节直接读取维护者已清洗的三任务数据，完成 Train / Validation / Holdout Test 划分、
泄漏审计和 Validation Baseline。原始 CSV 的清洗、分类、筛选及映射核验由维护者工具
`utils/prepare_official_dataset.py` 提前完成，学生无需运行。

`01_Synthetic_Data_Engineering.ipynb` 是完全可选的实验，各 Notebook 使用独立 Kernel，通过文件交接。

## 学习目标与课程边界

1. 核验维护者提供的已清洗数据，按固定种子完成 80% / 10% / 10% 的三集合划分。
2. 检查 ID、精确 Prompt、规范化 Prompt、结构指纹和 Numeral 答案直接暴露。
3. 保存数据集划分、Dataset Manifest、Leakage Audit Report。
4. 仅在 Validation 校准生成预算、执行 Baseline，分析 Accuracy、截断率和错误样本。

Holdout 只用于泄漏审计和未来最终评估，不参与预算选择或调参。
三类任务均计划参加后续 SFT；是否需要 RL 留待 SFT 后的实验判断，本轮不修改训练算法或 Reward。

## 1. 已清洗数据与配置

数据位于 `data/processed/tasksets/roman_gravity_equation_v1/real_cleaned.csv`。
维护者清洗结果及来源 SHA256 保存在 `official_preparation_manifest.json`；本节只读取并核验，不重写清洗结果。

划分和审计保留原版本路径。相同内容可复用，不同内容要求更换版本，避免覆盖历史产物。
GPU 阶段使用独立 Run ID，默认关闭，不安装依赖或下载模型。

In [ ]:
from __future__ import annotations
import hashlib, json, math, os, re, time, unicodedata
from collections import Counter, defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd()
dataset_version = os.environ.get("TASKSET_VERSION", "roman_gravity_equation_v1")
if not re.fullmatch(r"[A-Za-z0-9_-]+", dataset_version): raise ValueError("非法数据版本")
output_dir = project_root / "data/processed/tasksets" / dataset_version
cleaned_path = output_dir / "real_cleaned.csv"
preparation_path = output_dir / "official_preparation_manifest.json"
split_dir = output_dir / "splits"
output_dir.mkdir(parents=True, exist_ok=True)
split_dir.mkdir(exist_ok=True)
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
TASK_ORDER = ("numeral_system", "gravity", "equation_numeric")
TARGET_TASKS = set(TASK_ORDER)
SEED = 42
baseline_phase = os.environ.get("BASELINE_PHASE", "off")  # off / calibration / calibration_extension / evaluation
run_id = os.environ.get("BASELINE_RUN_ID", "roman_round1")
if baseline_phase not in {"off", "calibration", "calibration_extension", "evaluation"}: raise ValueError("非法 GPU 阶段")
if not re.fullmatch(r"[A-Za-z0-9_-]+", run_id): raise ValueError("非法 Run ID")
run_dir = output_dir / "baseline" / run_id
TEMPERATURE, TOP_P, TOP_K = 0.6, 0.95, 20
VLLM_MAX_MODEL_LEN = 20480  # 所有预算对照共享上下文容量
BUDGET_GRID = {"numeral_system": (512, 1024), "gravity": (4096, 8192, 16384), "equation_numeric": (4096, 8192, 16384)}
NUMERAL_EXTENSION_BUDGETS = (4096, 8192, 16384)  # only after both required short budgets failed
CALIBRATION_PER_TASK = 3
EVALUATION_PER_TASK = 50
INFERENCE_BATCH_SIZE = 10
OFFICIAL_VERIFIER_VERSION = "official_nvidia_metric_v3"

def sha256(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def write_once(path, payload):
    path = Path(path); data = payload if isinstance(payload, bytes) else str(payload).encode("utf-8")
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.read_bytes() != data: raise FileExistsError(f"拒绝覆盖既有版本产物: {path}")
    if not path.exists(): path.write_bytes(data)
def save_json_once(path, value): write_once(path, json.dumps(value, ensure_ascii=False, indent=2) + "\n")
def jsonl_text(rows): return "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows)

### 1.1 核验维护者提供的数据

只检查数据契约、内容哈希和已声明的题型，不读取原始 `train.csv`，也不重新分类或过滤样本。
若预处理产物缺失或版本不一致，应由维护者重新分发。

In [ ]:
if not cleaned_path.exists() or not preparation_path.exists():
    raise FileNotFoundError("缺少维护者提供的 real_cleaned.csv 或 official_preparation_manifest.json；无需学生清洗原始 CSV")
preparation_manifest = json.loads(preparation_path.read_text(encoding="utf-8"))
if preparation_manifest["tasks"] != list(TASK_ORDER):
    raise ValueError("已清洗数据的任务版本不一致")
if sha256(cleaned_path) != preparation_manifest["files"]["real_cleaned.csv"]:
    raise ValueError("已清洗数据与维护者 Manifest 不一致")
cleaned_df = pd.read_csv(cleaned_path, dtype=str, keep_default_na=False)
required_columns = ["source_row", "source_dataset", "source_sha256", "id", "prompt", "answer", "task_type", "equation_subtype"]
if not set(required_columns).issubset(cleaned_df.columns):
    raise ValueError(f"已清洗数据缺少字段：{set(required_columns) - set(cleaned_df.columns)}")
assert len(cleaned_df) == preparation_manifest["cleaned_rows"]
assert set(cleaned_df.task_type) == TARGET_TASKS
assert not cleaned_df[["id", "prompt", "answer", "task_type"]].eq("").any().any()
assert not cleaned_df.id.duplicated().any() and not cleaned_df.prompt.duplicated().any()
source_sha256 = preparation_manifest["source"]["sha256"]
assert cleaned_df.source_sha256.eq(source_sha256).all()
print(cleaned_df.groupby("task_type").size().rename("rows").to_frame())

## 2. Baseline 的 Numeral Verifier

标准范围采用 1–3999、减法写法 IV/IX/XL/XC/CD/CM。Encoder 按位值构造字符串；Verifier 以独立字符求和逻辑解析，
先用语法约束排除 IIII、IL、VX 等非规范拼写。以下代码服务于 Baseline 评分和机制测试。
官方数据逐条映射核验由维护者工具完成，本节不重复执行。

In [ ]:
ROMAN_GRAMMAR = re.compile(r"M{0,3}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})")
ROMAN_VALUES = {"I":1, "V":5, "X":10, "L":50, "C":100, "D":500, "M":1000}

def arabic_to_roman(number):
    if not isinstance(number, int) or isinstance(number, bool) or not 1 <= number <= 3999: raise ValueError("标准范围为 1..3999")
    thousands = ("", "M", "MM", "MMM")
    hundreds = ("", "C", "CC", "CCC", "CD", "D", "DC", "DCC", "DCCC", "CM")
    tens = ("", "X", "XX", "XXX", "XL", "L", "LX", "LXX", "LXXX", "XC")
    ones = ("", "I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX")
    return thousands[number//1000] + hundreds[number//100%10] + tens[number//10%10] + ones[number%10]

def canonical_roman_to_int(roman):
    if not isinstance(roman, str) or not roman or not ROMAN_GRAMMAR.fullmatch(roman): raise ValueError("非规范罗马数字")
    values = [ROMAN_VALUES[char] for char in roman]
    result = sum(-value if index+1 < len(values) and value < values[index+1] else value for index, value in enumerate(values))
    if not 1 <= result <= 3999: raise ValueError("超出标准范围")
    return result

def verify_roman_number(number, roman):
    if not re.fullmatch(r"[0-9]+", str(number)): return False
    try: return canonical_roman_to_int(roman) == int(number)
    except (TypeError, ValueError, KeyError): return False

def parse_numeral_prompt(prompt):
    pairs = [(int(a), b) for a,b in re.findall(r"(?m)^(\d+)\s*->\s*([A-Z]+)\s*$", prompt)]
    mapping_lines = [line for line in prompt.splitlines() if "->" in line]
    query = re.search(r"Now, write the number (\d+) in the Wonderland numeral system\.\s*$", prompt)
    if not query or not pairs or len(mapping_lines) != len(pairs): raise ValueError("Numeral 题面解析失败")
    return pairs, int(query.group(1))

### 2.1 Verifier 的 CPU 边界和负样本

穷举 1–3999 检查两个独立实现一致；错误拼写、错误数值及截断回答分别测试。
这种 CPU 确定性验证检查的是评测逻辑，不是模型 Baseline。

In [ ]:
roman_tests = []
for number in range(1,4000):
    assert verify_roman_number(number, arabic_to_roman(number))
for invalid in ("", "IIII", "VV", "IL", "IC", "VX", "IIV", "MMMM", "xiv", "12", "X X"):
    roman_tests.append({"case":"noncanonical", "value":invalid, "passed":not verify_roman_number(14,invalid)})
for number, roman in ((4,"VI"),(9,"XI"),(100,"XC")):
    roman_tests.append({"case":"wrong_number", "value":roman, "passed":not verify_roman_number(number,roman)})
assert all(row["passed"] for row in roman_tests)
roman_report = {"status":"passed", "verifier":"independent_canonical_parser", "exhaustive_roundtrip_cases":3999,
                "official_examples":preparation_manifest["numeral_mapping"]["example_count"],
                "official_query_answers":preparation_manifest["numeral_mapping"]["query_count"],
                "all_official_mappings_valid":preparation_manifest["numeral_mapping"]["all_valid"], "negative_cases":roman_tests}
save_json_once(output_dir / "numeral_verifier_tests.json", roman_report)

schema_tests = []
for number, roman, expected in [(1,"I",True),("001","I",True),(1.2,"I",False),(True,"I",False),(-1,"I",False),(0,"",False),(4000,"MMMM",False),(9,"VIIII",False)]:
    assert verify_roman_number(number,roman) == expected
    schema_tests.append({"arabic":number,"roman":roman,"expected_valid":expected,"passed":True})
save_json_once(output_dir/"numeral_schema_tests.json", {"status":"passed","cases":schema_tests})

## 3. 分层划分与分级泄漏审计

固定种子按任务独立洗牌；Validation/Test 各取最接近 10% 的整数，余下归 Train。
规范化只做 NFKC、空白与大小写统一，保留所有数值和运算符。结构指纹进一步掩去数值（Numeral 同时掩去罗马串），只表示模板相似。
本轮不以结构碰撞认定数据泄漏，也不宣称完成 Structural OOD。

In [ ]:
def prompt_hash(text): return hashlib.sha256(str(text).encode("utf-8")).hexdigest()
def normalize_prompt(text): return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(text)).strip()).casefold()
def structure_fingerprint(text, task_type):
    text = normalize_prompt(text)
    if task_type == "numeral_system": text = re.sub(r"\b[ivxlcdm]+\b", "<roman>", text)
    text = re.sub(r"\d{2}" if task_type == "equation_numeric" else r"[-+]?\d+(?:\.\d+)?", "<num>", text)
    return prompt_hash(text)

def make_stratified_split(frame, seed=SEED):
    rng = np.random.default_rng(seed); result = frame.copy(); result["split"] = ""
    for task, group in frame.groupby("task_type",sort=True):
        indices = group.index.to_numpy(copy=True); rng.shuffle(indices); n = len(indices)
        n_val = int(round(n*.1)); n_test = int(round(n*.1))
        if min(n_val,n_test,n-n_val-n_test) < 1: raise ValueError("集合规模不足")
        result.loc[indices[:n-n_val-n_test],"split"] = "train"
        result.loc[indices[n-n_val-n_test:n-n_test],"split"] = "validation"
        result.loc[indices[n-n_test:],"split"] = "holdout_test"
    return result

cleaned_df["prompt_hash"] = cleaned_df.prompt.map(prompt_hash)
cleaned_df["normalized_prompt_hash"] = cleaned_df.prompt.map(lambda text:prompt_hash(normalize_prompt(text)))
cleaned_df["structure_fingerprint"] = [structure_fingerprint(row.prompt,row.task_type) for row in cleaned_df.itertuples()]
split_df = make_stratified_split(cleaned_df)
assert split_df.split.ne("").all() and split_df.id.nunique() == len(cleaned_df)
for name, group in split_df.groupby("split",sort=True): write_once(split_dir / f"{name}.csv", group.to_csv(index=False))
split_counts = split_df.groupby(["split","task_type"]).size().unstack(fill_value=0)
print(split_counts)

In [ ]:
def cross_split_duplicate_counts(frame, key):
    ownership = frame.groupby(key)["split"].agg(lambda values: sorted(set(values)))
    collisions = ownership[ownership.map(len)>1]
    return {"duplicate_keys":len(collisions),"rows_in_cross_split_keys":int(frame[key].isin(collisions.index).sum()),
            "examples":[{"key":str(key),"splits":owners} for key,owners in collisions.head(5).items()]}

leakage_checks = {name:cross_split_duplicate_counts(split_df,key) for name,key in
                  (("exact_id","id"),("exact_prompt","prompt_hash"),("normalized_prompt","normalized_prompt_hash"),("structure_fingerprint","structure_fingerprint"))}
for key in ("exact_id","exact_prompt","normalized_prompt"): assert leakage_checks[key]["duplicate_keys"] == 0
save_json_once(output_dir / "structural_collision_note.json", {
    "interpretation":"structure similarity only; not proof of shared source or answer leakage",
    "preserved_information":"task words and Equation operators; numeric literal widths retained for Equation",
    "group_key":"unavailable", "structural_ood":"not_claimed", "counts":leakage_checks["structure_fingerprint"]})


### 3.1 Numeral 的答案直接暴露

除样本重复外，逐条检查 Query 的 `(阿拉伯数字, 罗马数字)` 是否出现在其他集合的示例中，另统计相同 Query 在其他集合目标中的重复。
这表示有限映射任务的答案暴露，**不等于重复完整样本**。相同映射在当前题内出现也单独记录。
仅报告 IID 暴露程度，不据此改动已规定的随机划分；未来需要数字值 OOD 时应另建实验协议。

In [ ]:
numeral_frame = split_df[split_df.task_type == "numeral_system"]
example_index, query_index = defaultdict(list), defaultdict(list)
for row in numeral_frame.itertuples():
    pairs, query = parse_numeral_prompt(row.prompt)
    for number, roman in pairs: example_index[(row.split,number,roman)].append(row.id)
    query_index[(row.split,query,row.answer)].append(row.id)
exposure_rows = []
for row in numeral_frame.itertuples():
    pairs, query = parse_numeral_prompt(row.prompt)
    for other in ("train","validation","holdout_test"):
        if other == row.split: continue
        matched = sorted(set(example_index.get((other,query,row.answer),[])))
        query_matched = sorted(set(query_index.get((other,query,row.answer),[])))
        exposure_rows.append({"id":row.id,"source_split":row.split,"other_split":other,"query_number":query,"query_roman":row.answer,
                              "answer_exposed_in_other_examples":bool(matched),"matching_example_prompt_ids":json.dumps(matched),
                              "same_query_in_other_targets":bool(query_matched),"other_target_prompt_ids":json.dumps(query_matched),
                              "query_already_in_own_examples":(query,row.answer) in pairs})
exposure_df = pd.DataFrame(exposure_rows)
write_once(output_dir / "numeral_cross_split_exposure.csv",exposure_df.to_csv(index=False))
exposure_summary = []
for (source,other), group in exposure_df.groupby(["source_split","other_split"],sort=True):
    exposure_summary.append({"source_split":source,"other_split":other,"queries":len(group),
        "exposed_in_other_examples":int(group.answer_exposed_in_other_examples.sum()),
        "exposure_rate":float(group.answer_exposed_in_other_examples.mean()),
        "same_query_in_other_targets":int(group.same_query_in_other_targets.sum())})
leakage_report = {"schema_version":"three-task-leakage/v1", "sample_duplicate_audit":"passed",
    "checks":leakage_checks,"numeral_answer_exposure":exposure_summary,
    "numeral_intra_prompt_exposure":int(exposure_df.drop_duplicates("id").query_already_in_own_examples.sum()),
    "interpretation":{"real_duplicates":"ID / exact / normalized prompt", "direct_answer_exposure":"same numeral query mapping in another split's examples",
                      "structural_collision":"similarity indicator only"},
    "no_leakage_claim":False, "limitation":"IID numeral mappings repeat across splits; unknown shared rule/seed groups"}
save_json_once(output_dir / "leakage_audit_report.json",leakage_report)
print(pd.DataFrame(exposure_summary))


### 3.2 Manifest、统计和数据图

Manifest 绑定原始 CSV、分类、清洗与三个划分文件的 SHA256。后续模型实验保存 Manifest 哈希，避免误用旧版 Bit 数据。
这里只建立新数据接口；冻结的后训练 Notebook 尚未切换到新数据版本，不应直接将旧 Checkpoint 结果当作新任务结果。

In [ ]:
dataset_manifest = {"schema_version":"nemotron-taskset/v3", "dataset_version":dataset_version,"seed":SEED,
    "tasks":list(TASK_ORDER),"difficulty":{"numeral_system":"easy","gravity":"medium","equation_numeric":"hard"},
    "ratios_requested":{"train":.8,"validation":.1,"holdout_test":.1}, "method":"per-task IID, round 10% for validation and holdout",
    "source":preparation_manifest["source"],"files":{},"artifacts":{},
    "baseline_split":"validation", "holdout_policy":"audit_and_future_final_evaluation_only", "bit_in_default_dataset":False,
    "leakage_report":str((output_dir/"leakage_audit_report.json").relative_to(project_root)),
    "all_tasks_participate_in_sft":True, "rl_task_decision":"after SFT; not changed in this round"}
for name in ("train","validation","holdout_test"):
    path = split_dir / f"{name}.csv";group = split_df[split_df.split==name]
    dataset_manifest["files"][name] = str(path.relative_to(project_root))
    dataset_manifest["artifacts"][name] = {"sha256":sha256(path),"rows":len(group),"by_task":group.task_type.value_counts().to_dict()}
for filename in ("real_cleaned.csv","classification_audit.csv","numeral_mapping_audit.csv","leakage_audit_report.json"):
    dataset_manifest.setdefault("other_artifacts",{})[filename] = sha256(output_dir/filename)
save_json_once(output_dir / "dataset_manifest.json",dataset_manifest)
save_json_once(output_dir / "dataset_statistics.json", {"raw":preparation_manifest["raw"],"classification_counts":preparation_manifest["classification_counts"],
    "cleaned_rows":len(cleaned_df),"cleaned_by_task":cleaned_df.task_type.value_counts().to_dict(),
    "split_counts":split_counts.to_dict(orient="index"),"unknown_rows":preparation_manifest["unknown_rows"]})
equation_subtype_counts = {
    name: split_df[(split_df.split==name)&(split_df.task_type=="equation_numeric")].equation_subtype.value_counts().to_dict()
    for name in ("train", "validation", "holdout_test")}
save_json_once(output_dir/"equation_subtype_statistics.json", {
    "source":"exact Query operator membership in example operators", "counts":equation_subtype_counts,
    "used_for_filtering_or_budget_selection":False})
fig, ax = plt.subplots(figsize=(8,4));split_counts.plot.bar(ax=ax)
ax.set(title="Three-task stratified IID split",xlabel="Split",ylabel="Number of samples");ax.legend(title="Task")
fig.tight_layout()
if not (output_dir/"split_task_distribution.png").exists():fig.savefig(output_dir/"split_task_distribution.png",dpi=150)
plt.show()

## 4. 统一完成状态、提取与比赛 Verifier

保留 `official_nvidia_metric_v3`：`length` 或非正常完成不能计正确；数值容差为 `rel_tol=1e-2, abs_tol=1e-5`。
新增 Roman 分支要求规范拼写，并独立检查其数值是否对应 Prompt 中的 Query；不依赖模型自评。
若保留了 `</think>`，只在其后的最终回答区提取；绝不从截断 Thinking 取中间正确数字。

In [ ]:
def normalize_answer_text(value):
    text = str(value).strip()
    # Remove paired presentation wrappers, preserving literal operators (*, $, `).
    for wrapper in ("```", "**", "$$", "`", "$"):
        if text.startswith(wrapper) and text.endswith(wrapper) and len(text) > 2*len(wrapper):
            text = text[len(wrapper):-len(wrapper)].strip()
    text = re.sub(r"^```(?:text|json)?\s*|\s*```$","",text,flags=re.I).strip()
    for _ in range(3): text = re.sub(r"\\(?:text|mathrm|mathbf)\{([^{}]*)\}", r"\1", text)
    text = re.sub(r"\\([{}#$%&_])", r"\1", text)
    return text.strip()

def boxed_values(text):
    values=[]
    for match in re.finditer(r"\\boxed\{",text):
        start=match.end();depth=1;end=start
        while end < len(text) and depth:
            if text[end]=="{" and (end==0 or text[end-1]!="\\"):depth+=1
            elif text[end]=="}" and (end==0 or text[end-1]!="\\"):depth-=1
            end+=1
        if depth==0:values.append(text[start:end-1])
    return values

def extract_final_answer(text,task_type):
    text=str(text).rsplit("</think>",1)[-1].replace("<|im_end|>","").replace("<|endoftext|>","")
    boxed=boxed_values(text)
    if boxed:return normalize_answer_text(boxed[-1]),"boxed"
    tagged=re.findall(r"<(?P<tag>final|answer)>\s*(?P<value>.*?)\s*</(?P=tag)>",text,flags=re.I|re.S)
    if tagged:return normalize_answer_text(tagged[-1][1]),"tagged"
    lines=[line.strip() for line in text.splitlines() if line.strip()]
    for line in reversed(lines):
        labeled=re.match(r"(?i)^(?:final answer|answer|最终答案|最终输出)\s*[:：]\s*(.+)$",line.strip("* "))
        if labeled:return normalize_answer_text(labeled.group(1)),"final_label"
    if lines:
        candidate=normalize_answer_text(lines[-1]);pattern=r"[IVXLCDM]+" if task_type=="numeral_system" else (r"(?:[-+]?\d+(?:\.\d+)?|[^\w\s]\d+|\d+[^\w\s])" if task_type=="equation_numeric" else r"[-+]?\d+(?:\.\d+)?")
        if re.fullmatch(pattern,candidate):return candidate,"standalone_final_line"
    return None,"unmarked_or_empty"

def official_numeric_match(predicted,expected):
    try:p,e=float(predicted),float(expected)
    except (TypeError,ValueError):return False
    return math.isfinite(p) and math.isfinite(e) and math.isclose(p,e,rel_tol=1e-2,abs_tol=1e-5)

def verify_answer(task_type,predicted,expected,query_number=None):
    if predicted is None:return False,"not_extracted"
    p,e=normalize_answer_text(predicted),normalize_answer_text(expected)
    if task_type=="numeral_system":
        try:
            value=canonical_roman_to_int(p);target=canonical_roman_to_int(e)
            return value==target and (query_number is None or value==query_number),"canonical_roman_independent_parser"
        except ValueError:return False,"invalid_roman_format"
    if task_type in {"gravity","equation_numeric"}:
        pattern = r"(?:[-+]?\d+(?:\.\d+)?|[^\w\s]\d+|\d+[^\w\s])" if task_type=="equation_numeric" else r"[-+]?\d+(?:\.\d+)?"
        if not re.fullmatch(pattern,p):return False,"invalid_numeric_format"
        # Numeric24 includes literal operator prefixes/suffixes. Retain exact text fallback.
        return official_numeric_match(p,e) or p==e,"official_math_isclose_or_exact_text"
    raise ValueError("非本轮任务")

def verify_completed_output(task,prompt,output,expected,finish_reason):
    provisional,method=extract_final_answer(output,task)
    if finish_reason!="stop":return {"extracted_answer":None,"provisional_extracted_answer":provisional,"extraction_success":False,
        "format_valid":False,"correct":False,"extraction_method":method,"verification_method":"truncated_output" if finish_reason=="length" else "incomplete_output"}
    query=parse_numeral_prompt(prompt)[1] if task=="numeral_system" else None
    correct,verification=verify_answer(task,provisional,expected,query)
    return {"extracted_answer":provisional,"provisional_extracted_answer":provisional,"extraction_success":provisional is not None,
        "format_valid":provisional is not None and not verification.startswith("invalid_"),"correct":bool(correct),
        "extraction_method":method,"verification_method":verification}


In [ ]:
completion_tests=[]
for task,prompt,output,expected,finish,correct in [
 ("numeral_system",cleaned_df[cleaned_df.task_type=="numeral_system"].iloc[0].prompt,r"\boxed{\text{XXXVIII}}","XXXVIII","stop",True),
 ("numeral_system",cleaned_df[cleaned_df.task_type=="numeral_system"].iloc[0].prompt,r"\boxed{XXXVIII}","XXXVIII","length",False),
 ("numeral_system",cleaned_df[cleaned_df.task_type=="numeral_system"].iloc[0].prompt,r"\boxed{IIII}","XXXVIII","stop",False),
 ("gravity","CPU test",r"\boxed{24.28}","24.28","length",False),
 ("gravity","CPU test",r"\boxed{24.28}","24.28","stop",True),
 ("equation_numeric","CPU test",r"\boxed{32}","32","stop",True)]:
    # Construct the positive numeral example from the actual first query, not a presumed source row.
    if task=="numeral_system" and correct:
        expected=arabic_to_roman(parse_numeral_prompt(prompt)[1]);output=r"\boxed{\text{"+expected+"}}"
    result=verify_completed_output(task,prompt,output,expected,finish)
    assert result["correct"]==correct,(task,result)
    completion_tests.append({"task":task,"finish_reason":finish,"expected_correct":correct,"passed":True})
assert extract_final_answer("we are still checking 24.28", "gravity")[0] is None
assert extract_final_answer("<think>\\boxed{IV}</think>still checking", "numeral_system")[0] is None
literal_equation_cases = []
for answer in cleaned_df[cleaned_df.task_type=="equation_numeric"].answer.unique():
    result = verify_completed_output("equation_numeric", "CPU test", "Final Answer: "+answer, answer, "stop")
    assert result["correct"], (answer,result)
    literal_equation_cases.append({"answer":answer,"passed":True})
assert not verify_completed_output("equation_numeric","CPU test",r"\boxed{85}","$85","stop")["correct"]
save_json_once(output_dir/"equation_answer_format_tests.json", {"status":"passed","cases":literal_equation_cases})
save_json_once(output_dir/"completion_verifier_tests_v2.json",{"status":"passed","verifier_version":OFFICIAL_VERIFIER_VERSION,"cases":completion_tests})

format_regressions=[]
for output, expected in [(r"\boxed{\{17}","{17"),(r"\boxed{\$85}","$85"),(r"\boxed{*53}","*53"),("Final Answer: 04`","04`"),("Final Answer: \\93","\\93")]:
    result=verify_completed_output("equation_numeric","CPU test",output,expected,"stop")
    assert result["correct"], (output,result)
    format_regressions.append({"raw_output":output,"expected":expected,"passed":True})
save_json_once(output_dir/"answer_extraction_regression_tests.json", {"status":"passed","cases":format_regressions})


## 5. Validation 预算校准与固定评测样本

每类固定随机选择 3 条校准题，所有预算重复同一组 Prompt；另选不重叠的 50 条 Validation 评测题。
所有对照使用相同 T=0.6、top-p=0.95、top-k=20、seed=42、上下文上限 20480，不改题面、不注入先验。
预算选择只看完成与格式，不用正确率选择：优先满足全部校准题正常完成且格式有效的最小档；否则保留最高档并标记上限未解决。绝不自动超过 16384。

In [ ]:
calibration_frames=[];evaluation_frames=[]
for index,task in enumerate(TASK_ORDER):
    group=split_df[(split_df.split=="validation")&(split_df.task_type==task)].sort_values("id")
    shuffled=group.sample(frac=1,random_state=SEED+index)
    calibration_frames.append(shuffled.iloc[:CALIBRATION_PER_TASK])
    evaluation_frames.append(shuffled.iloc[CALIBRATION_PER_TASK:CALIBRATION_PER_TASK+EVALUATION_PER_TASK])
calibration_frame=pd.concat(calibration_frames,ignore_index=True)
evaluation_frame=pd.concat(evaluation_frames,ignore_index=True)
assert set(calibration_frame.id).isdisjoint(evaluation_frame.id)
assert evaluation_frame.groupby("task_type").size().eq(EVALUATION_PER_TASK).all()
assert evaluation_frame.split.eq("validation").all()
protocol={"dataset_manifest_sha256":sha256(output_dir/"dataset_manifest.json"),"model_id":MODEL_ID,
    "calibration_ids":calibration_frame.groupby("task_type").id.apply(list).to_dict(),
    "evaluation_ids":evaluation_frame.groupby("task_type").id.apply(list).to_dict(),
    "temperature":TEMPERATURE,"top_p":TOP_P,"top_k":TOP_K,"seed":SEED,"max_model_len":VLLM_MAX_MODEL_LEN,
    "budget_grid":{key:list(values) for key,values in BUDGET_GRID.items()},"evaluation_per_task":EVALUATION_PER_TASK,
    "evaluation_split":"validation","holdout_used_for_budget_selection":False,"calibration_evaluation_disjoint":True,
    "prompt":"original official prompt + official Qwen Chat Template, Thinking enabled; no priors / SFT / synthetic"}
save_json_once(output_dir/"baseline_protocol.json",protocol)
print({"calibration":calibration_frame.groupby("task_type").size().to_dict(),"evaluation":evaluation_frame.groupby("task_type").size().to_dict()})


### 5.1 GPU 监控与真实输出记录

vLLM 在独立官方 Profile 运行；模型每阶段只加载一次。设备显存用 nvidia-smi 采样，另保存 PyTorch allocator 峰值，二者不混用。
每批返回后立即写 JSONL。单样本摊销时间和请求实际延迟分别标注；吞吐由批量总生成 Token / 批量 Wall Time 计算。

In [ ]:
def render_qwen3_prompt(tokenizer,prompt):
    return tokenizer.apply_chat_template([{"role":"user","content":prompt}],tokenize=False,add_generation_prompt=True,enable_thinking=True)

def _sample_nvidia_smi_memory(stop,samples):
    import subprocess
    while not stop.is_set():
        try:
            data=subprocess.check_output(["nvidia-smi","--query-gpu=memory.used","--format=csv,noheader,nounits","-i","0"],text=True,timeout=3)
            samples.append(float(data.strip().splitlines()[0]))
        except (OSError,ValueError,subprocess.SubprocessError):pass
        stop.wait(.25)

def result_record(original,request,elapsed,batch_size,budget,phase,batch_id,think_end):
    generated=request.outputs[0];ids=list(generated.token_ids or []);prompt_ids=list(request.prompt_token_ids or [])
    score=verify_completed_output(original.task_type,original.prompt,generated.text,original.answer,generated.finish_reason)
    metrics=getattr(request,"metrics",None)
    first=getattr(metrics,"first_token_time",None);last=getattr(metrics,"last_token_time",None);arrival=getattr(metrics,"arrival_time",None)
    return {"id":original.id,"source_row":int(original.source_row),"task_type":original.task_type,"source_split":"validation",
        "dataset_version":dataset_version,"dataset_manifest_sha256":protocol["dataset_manifest_sha256"],"model_id":MODEL_ID,
        "phase":phase,"batch_id":batch_id,"prompt":original.prompt,"expected_answer":original.answer,"raw_output":generated.text,
        **score,"verifier_version":OFFICIAL_VERIFIER_VERSION,"finish_reason":generated.finish_reason,"stop_reason":generated.stop_reason,
        "truncated":generated.finish_reason=="length","prompt_token_ids":prompt_ids,"completion_token_ids":ids,
        "prompt_tokens":len(prompt_ids),"generation_tokens":len(ids),"max_new_tokens":budget,
        "temperature":TEMPERATURE,"top_p":TOP_P,"top_k":TOP_K,"seed":SEED,
        "think_end_token_seen":think_end in ids,"think_end_index":ids.index(think_end) if think_end in ids else None,
        "batch_wall_time_seconds":elapsed,"batch_size":batch_size,"amortized_inference_time_seconds":elapsed/batch_size,
        "request_latency_seconds":last-arrival if last is not None and arrival is not None else None,
        "time_to_first_token_seconds":first-arrival if first is not None and arrival is not None else None}

def metric_group(rows):
    n=len(rows);elapsed=sum(row["amortized_inference_time_seconds"] for row in rows);tokens=sum(row["generation_tokens"] for row in rows)
    return {"records":n,"accuracy":sum(row["correct"] for row in rows)/n,
        "answer_extraction_success_rate":sum(row["extraction_success"] for row in rows)/n,
        "invalid_output_rate":sum(not row["format_valid"] for row in rows)/n,
        "truncation_rate":sum(row["truncated"] for row in rows)/n,"average_generation_tokens":tokens/n,
        "average_amortized_inference_time_seconds":elapsed/n,"generation_wall_time_seconds":elapsed,
        "output_tokens":tokens,"output_token_throughput":tokens/elapsed if elapsed else 0.0}

def summarize_baseline_records(rows):
    if not rows:return {"records":0,"status":"no_records"}
    by_task={task:metric_group([row for row in rows if row["task_type"]==task]) for task in TASK_ORDER if any(row["task_type"]==task for row in rows)}
    return {**metric_group(rows),"overall_accuracy_sample_weighted":sum(row["correct"] for row in rows)/len(rows),
        "macro_accuracy":sum(row["accuracy"] for row in by_task.values())/len(by_task),"by_task":by_task}


### 5.2 批量执行与持久化

同一校准题在每档预算都执行，不能把“只续测失败题”当作同一评测分母。
评测只读取已经保存的预算决策。所有结果先核对 Validation ID、原始 Prompt 和答案，禁止混入旧版数据或 Adapter。

In [ ]:
def choose_task_budgets(rows, budget_grid=None):
    budget_grid = BUDGET_GRID if budget_grid is None else budget_grid
    decisions={}
    for task in TASK_ORDER:
        trials=[]
        for budget in budget_grid[task]:
            group=[row for row in rows if row["task_type"]==task and row["max_new_tokens"]==budget]
            if len(group)!=CALIBRATION_PER_TASK:raise ValueError("校准记录不完整")
            valid=all(row["finish_reason"]=="stop" and row["format_valid"] for row in group)
            trials.append({"max_new_tokens":budget,"all_complete_and_valid":valid,**metric_group(group)})
        qualifying=[row["max_new_tokens"] for row in trials if row["all_complete_and_valid"]]
        decisions[task]={"max_new_tokens":min(qualifying) if qualifying else max(budget_grid[task]),
                         "status":"calibrated" if qualifying else "budget_cap_unresolved","trials":trials}
    return {"status":"completed","selection_rule":"smallest budget completing all fixed calibration prompts with valid format; else highest allowed cap",
        "task_budgets":decisions,"dataset_manifest_sha256":protocol["dataset_manifest_sha256"],"protocol_sha256":sha256(output_dir/"baseline_protocol.json"),
        "decode":{"temperature":TEMPERATURE,"top_p":TOP_P,"top_k":TOP_K,"seed":SEED},"holdout_used":False}

def _execute_batches(engine,tokenizer,plan,phase,path):
    from vllm import SamplingParams
    rows=[];think_end=tokenizer.convert_tokens_to_ids("</think>")
    lookup=split_df[split_df.split=="validation"].set_index("id")
    for index,(frame,budget) in enumerate(plan):
        for row in frame.itertuples():
            reference=lookup.loc[row.id]
            if row.prompt!=reference.prompt or row.answer!=reference.answer:raise ValueError("不是原始 Validation 实例")
        rendered=[render_qwen3_prompt(tokenizer,prompt) for prompt in frame.prompt]
        if any(len(tokenizer.encode(text,add_special_tokens=False))+budget>VLLM_MAX_MODEL_LEN for text in rendered):raise ValueError("上下文长度不足")
        params=SamplingParams(temperature=TEMPERATURE,top_p=TOP_P,top_k=TOP_K,seed=SEED,max_tokens=budget,skip_special_tokens=False,
                              repetition_penalty=1.0,presence_penalty=0.0,frequency_penalty=0.0)
        started=time.perf_counter();outputs=engine.generate(rendered,params,use_tqdm=True);elapsed=time.perf_counter()-started
        if len(outputs)!=len(frame):raise ValueError("返回记录数不一致")
        batch=[result_record(row,out,elapsed,len(frame),budget,phase,index,think_end) for row,out in zip(frame.itertuples(),outputs)]
        rows.extend(batch)
        with path.open("a",encoding="utf-8") as stream:stream.write(jsonl_text(batch))
        (path.parent/f"{phase}_progress.json").write_text(json.dumps({"status":"running","finished_records":len(rows),"completed_batches":index+1},indent=2))
        print("BATCH_SAVED",phase,index,len(rows),flush=True)
    return rows

def _inference_plan(phase):
    if phase=="calibration":
        return [(calibration_frame[calibration_frame.task_type==task].copy(),budget) for task in TASK_ORDER for budget in BUDGET_GRID[task]]
    if phase=="calibration_extension":
        original = json.loads((run_dir/"budget_decision.json").read_text())
        if original["task_budgets"]["numeral_system"]["status"]!="budget_cap_unresolved":
            raise ValueError("短预算已完成，不应追加 Numeral 校准")
        extension_protocol = {"base_protocol_sha256":sha256(output_dir/"baseline_protocol.json"),
            "task":"numeral_system","ids":calibration_frame[calibration_frame.task_type=="numeral_system"].id.tolist(),
            "budget_grid":list(NUMERAL_EXTENSION_BUDGETS),"reason":"both 512 and 1024 exhausted without final answer",
            "temperature":TEMPERATURE,"top_p":TOP_P,"top_k":TOP_K,"seed":SEED,"max_model_len":VLLM_MAX_MODEL_LEN,
            "selection_uses_accuracy":False,"upper_cap":16384}
        save_json_once(run_dir/"calibration_extension_protocol.json",extension_protocol)
        return [(calibration_frame[calibration_frame.task_type=="numeral_system"].copy(),budget) for budget in NUMERAL_EXTENSION_BUDGETS]
    decision_path=run_dir/"final_budget_decision.json"
    if not decision_path.exists(): decision_path=run_dir/"budget_decision.json"
    decision=json.loads(decision_path.read_text())
    if decision["protocol_sha256"]!=sha256(output_dir/"baseline_protocol.json"):raise ValueError("校准协议不匹配")
    plan=[]
    for task in TASK_ORDER:
        group=evaluation_frame[evaluation_frame.task_type==task];budget=decision["task_budgets"][task]["max_new_tokens"]
        for start in range(0,len(group),INFERENCE_BATCH_SIZE):plan.append((group.iloc[start:start+INFERENCE_BATCH_SIZE],budget))
    return plan


### 5.3 模型加载、Preflight、OOM 与阶段状态

推理只加载未训练的 BF16 Base Model，不启用 LoRA。进程退出后确认显存释放，再运行下一阶段。
异常或中断保留已落盘的完整批次；部分结果必须标为 `partial`，不能称为完成 Baseline。

In [ ]:
def run_gpu_phase(phase):
    if phase=="off":return {"status":"skipped_by_default"}
    import gc,threading,traceback,importlib.metadata,torch
    from transformers import AutoTokenizer
    from vllm import LLM,SamplingParams
    run_dir.mkdir(parents=True,exist_ok=True)
    path=run_dir/f"{phase}_results.jsonl";status_path=run_dir/f"{phase}_status.json"
    if path.exists() or status_path.exists():raise FileExistsError("阶段目录已有结果，请设置新的 BASELINE_RUN_ID，或在独立阶段目录检查失败记录")
    if not torch.cuda.is_available():raise RuntimeError("CUDA 不可用")
    free,total=torch.cuda.mem_get_info()
    preflight={"device":torch.cuda.get_device_name(0),"free_mib":free/2**20,"total_mib":total/2**20,"torch_cuda":torch.version.cuda,
               "versions":{name:importlib.metadata.version(name) for name in ("torch","transformers","vllm")},"fixture":False}
    save_json_once(run_dir/f"{phase}_preflight.json",preflight)
    if free/total<.85:raise RuntimeError("GPU 被占用，不终止其他任务")
    source=os.environ.get("QWEN3_MODEL_PATH",os.environ.get("LOCAL_MODEL_PATH",""))
    if not Path(source).is_dir() or not source:raise FileNotFoundError("请指定现有本地 Qwen3 权重，不自动下载")
    plan=_inference_plan(phase);engine=None;samples=[];stop=threading.Event()
    monitor=threading.Thread(target=_sample_nvidia_smi_memory,args=(stop,samples),daemon=True)
    started=time.perf_counter();torch.cuda.reset_peak_memory_stats();monitor.start()
    try:
        tokenizer=AutoTokenizer.from_pretrained(source,local_files_only=True)
        load_started=time.perf_counter()
        engine=LLM(model=source,tokenizer=source,dtype="bfloat16",tensor_parallel_size=1,generation_config="vllm",seed=SEED,
                   gpu_memory_utilization=float(os.environ.get("VLLM_GPU_MEMORY_UTILIZATION","0.85")),
                   max_model_len=VLLM_MAX_MODEL_LEN,max_num_seqs=16,enforce_eager=True)
        load_seconds=time.perf_counter()-load_started
        warm_started=time.perf_counter()
        engine.generate([render_qwen3_prompt(tokenizer,calibration_frame.iloc[0].prompt)],
            SamplingParams(temperature=TEMPERATURE,top_p=TOP_P,top_k=TOP_K,seed=SEED,max_tokens=32),use_tqdm=False)
        warmup_seconds=time.perf_counter()-warm_started
        rows=_execute_batches(engine,tokenizer,plan,phase,path)
        summary=summarize_baseline_records(rows)
        expected=sum(len(frame) for frame,_ in plan)
        if len(rows)!=expected:raise ValueError("未完成全部计划样本")
        if phase=="calibration":save_json_once(run_dir/"budget_decision.json",choose_task_budgets(rows))
        elif phase=="calibration_extension":
            original_rows = [json.loads(line) for line in (run_dir/"calibration_results.jsonl").read_text().splitlines() if line]
            # Re-score preserved outputs with operator-aware v3 extraction, never rerun short budgets.
            for record in original_rows:
                record.update(verify_completed_output(record["task_type"],record["prompt"],record["raw_output"],record["expected_answer"],record["finish_reason"]))
            grid={**BUDGET_GRID,"numeral_system":tuple(BUDGET_GRID["numeral_system"])+NUMERAL_EXTENSION_BUDGETS}
            decision=choose_task_budgets(original_rows+rows,grid)
            decision["extension_protocol_sha256"]=sha256(run_dir/"calibration_extension_protocol.json")
            decision["original_results_sha256"]=sha256(run_dir/"calibration_results.jsonl")
            save_json_once(run_dir/"final_budget_decision.json",decision)
        else:
            examples=[]
            for task in TASK_ORDER:
                for label,predicate in [("correct",lambda r:r["correct"]),("wrong_completed",lambda r:not r["correct"] and not r["truncated"]),("truncated",lambda r:r["truncated"])]:
                    chosen=next((row for row in rows if row["task_type"]==task and predicate(row)),None)
                    if chosen:examples.append({"case_type":label,**chosen})
            save_json_once(run_dir/"typical_cases.json",examples)
        result={"status":"completed","phase":phase,**summary,"load_seconds":load_seconds,"warmup_seconds":warmup_seconds,
                "model_id":MODEL_ID,"preflight":preflight,"protocol":protocol,"results_sha256":sha256(path),
                "model_config_sha256":sha256(Path(source)/"config.json"),"weights_dtype":"BF16","adapter":None,
                "budget_decision_path":str((run_dir/("final_budget_decision.json" if (run_dir/"final_budget_decision.json").exists() else "budget_decision.json")).relative_to(project_root))}
    except BaseException as error:
        message=traceback.format_exc();(run_dir/f"{phase}_error.log").write_text(message)
        count=sum(1 for line in path.open() if line.strip()) if path.exists() else 0
        result={"status":"partial" if count else "failed","failure_type":"oom" if "out of memory" in str(error).lower() else type(error).__name__,
                "completed_records":count,"error":repr(error),"phase":phase,"fixture":False}
    finally:
        stop.set();monitor.join(timeout=4)
        if engine is not None:del engine
        gc.collect();torch.cuda.empty_cache()
    result.update({"wall_time_seconds":time.perf_counter()-started,"gpu_device_peak_memory_mib":max(samples) if samples else None,
                   "peak_torch_allocated_mib":torch.cuda.max_memory_allocated()/2**20,
                   "resource_scope":"entire phase incl model load and warmup; device memory sampled every 0.25s"})
    save_json_once(status_path,result)
    return result


### 5.4 独立阶段执行入口

先运行 `BASELINE_PHASE=calibration`。本轮 Numeral 的 512/1024 均未完成，因此增加独立的
`calibration_extension`（4096/8192/16384），固定原三题和全部采样参数，不覆盖原校准。
正式 `evaluation` 优先读取带来源哈希的 `final_budget_decision.json`，每类运行另外 50 条 Validation。
补测最高 16384；达到上限仍失败则保留失败状态，不继续加长。

在官方 vLLM 环境中设置 `QWEN3_MODEL_PATH` 与唯一 `BASELINE_RUN_ID`，先设置 `BASELINE_PHASE=calibration` 执行本 Notebook。
结束进程、确认 GPU 释放后，用同一 Run ID 设置 `BASELINE_PHASE=evaluation` 再执行。
同一数据版本可安全复用；历史 GPU 文件不能覆盖。默认 `off` 只运行 CPU 数据和机制检查。

In [ ]:
baseline_status=run_gpu_phase(baseline_phase)
print(json.dumps(baseline_status,ensure_ascii=False,indent=2))


## 6. 结果复核、限制与后续问题

报告中 Overall 为样本加权，Macro 为三个任务准确率平均；本轮均衡每类 50 条时二者应一致。
Invalid Output 包含截断及不规范格式，错误但格式正确的答案另计错误。校准的 3 条不混入正式 50 条指标。

- Numeral 的高准确率是否部分来自易见映射及跨集合答案暴露？必须结合审计解释。
- Gravity 的主要失败是输出预算、估计误差还是执行错误？
- Equation 已正常结束但答错时，增加 Token 不能替代规则与全局约束学习。
- 三任务都参加后续 SFT；RL 是否必要留到 SFT 后用同一协议判断，本轮不改训练算法或 Reward。

In [ ]:
expected_artifacts=[output_dir/name for name in ("real_cleaned.csv","dataset_manifest.json","dataset_statistics.json","leakage_audit_report.json","numeral_verifier_tests.json","baseline_protocol.json")]
assert all(path.exists() for path in expected_artifacts)
assert sha256(cleaned_path) == preparation_manifest["files"]["real_cleaned.csv"]
cpu_report={"status":"passed","dataset_version":dataset_version,"rows":len(cleaned_df),"tasks":list(TASK_ORDER),
            "by_task":cleaned_df.task_type.value_counts().to_dict(),"numeral_verifier":"passed","completion_rules":"passed",
            "no_bit_rows":not cleaned_df.task_type.eq("bit_manipulation").any(),"raw_sha256":source_sha256,"gpu_enabled":baseline_phase!="off"}
save_json_once(output_dir/"cpu_smoke_report.json",{**cpu_report,"gpu_enabled":False,"scope":"CPU data and mechanisms only; GPU status stored separately"})
if (run_dir/"evaluation_status.json").exists():
    actual=json.loads((run_dir/"evaluation_status.json").read_text());print({key:actual.get(key) for key in ("status","records","overall_accuracy_sample_weighted","macro_accuracy","truncation_rate","gpu_device_peak_memory_mib")})
print(cpu_report)

## 7. 从持久化结果复核 Baseline

阶段状态必须为 `completed`，且每类恰好完成计划的 50 条原始 Validation，才能报告本轮 Baseline。
下列单元直接重算原始输出的完成状态和准确率，核对记录哈希；不加载模型，也不改写原始回答。
柱状图同时展示准确率与截断率，避免把未完成回答全部归因于规则能力。
对 Roman 还应结合有限映射的跨集合暴露报告分析，不能直接声称 SFT/RL 有必要或无必要。


In [ ]:
evaluation_status_path = run_dir / "evaluation_status.json"
if evaluation_status_path.exists():
    completed = json.loads(evaluation_status_path.read_text())
    if completed["status"] == "completed":
        records_path = run_dir / "evaluation_results.jsonl"
        assert sha256(records_path) == completed["results_sha256"]
        records = [json.loads(line) for line in records_path.read_text().splitlines() if line]
        assert Counter(row["task_type"] for row in records) == {task:EVALUATION_PER_TASK for task in TASK_ORDER}
        assert {row["id"] for row in records} == set(evaluation_frame.id)
        for row in records:
            assert len(row["completion_token_ids"]) == row["generation_tokens"] <= row["max_new_tokens"]
            assert len(row["prompt_token_ids"]) == row["prompt_tokens"]
            assert all(isinstance(token,int) for token in row["completion_token_ids"] + row["prompt_token_ids"])
            if row["truncated"]: assert row["generation_tokens"] == row["max_new_tokens"]
            checked = verify_completed_output(row["task_type"], row["prompt"], row["raw_output"], row["expected_answer"], row["finish_reason"])
            assert all(checked[key] == row[key] for key in ("correct", "extracted_answer", "format_valid", "extraction_success"))
        save_json_once(RUN_DIR / "raw_output_audit.json", {
            "status":"passed", "records":len(records), "fixture":False,
            "original_validation_ids_exact_match":True, "calibration_disjoint":True,
            "result_sha256_verified":True, "scores_recomputed_from_raw_outputs":True,
            "token_id_lengths_and_budgets_verified":True,
            "finish_reason_counts":dict(Counter(row["finish_reason"] for row in records)),
            "completed_without_think_end":sum(row["finish_reason"]=="stop" and not row["think_end_token_seen"] for row in records)})
        summary = summarize_baseline_records(records)
        save_json_once(run_dir / "baseline_summary.json", {"status":"completed", **summary,
            "gpu_device_peak_memory_mib":completed["gpu_device_peak_memory_mib"],
            "results_sha256":completed["results_sha256"], "evaluation_split":"validation"})
        metrics_frame = pd.DataFrame(summary["by_task"]).T
        print(metrics_frame.to_string())
        fig, ax = plt.subplots(figsize=(8,4))
        metrics_frame[["accuracy", "truncation_rate"]].plot.bar(ax=ax, ylim=(0,1))
        ax.set(title="Validation Baseline: accuracy and truncation", xlabel="Task", ylabel="Rate")
        ax.legend(title="Metric"); fig.tight_layout()
        if not (run_dir / "baseline_accuracy_truncation.png").exists():
            fig.savefig(run_dir / "baseline_accuracy_truncation.png", dpi=150)
        plt.show()
    else:
        print({"baseline":"incomplete", "stage_status":completed["status"], "details":completed})
else:
    print({"baseline":"Not Run", "reason":"本 Run ID 尚无正式评测状态；CPU 检查不替代 GPU 实验"})
